# ASR Meeting Transcript Pipeline — Kaggle T4 x2

**Pipeline:** normalize → denoise (DeepFilterNet) → VAD (Silero) → LID → ASR routing → summarize

| Language | Model |
|---|---|
| Tiếng Việt (conf ≥ 0.75) | Zipformer (`csukuangfj/sherpa-onnx-zipformer-vi-int8-2025-04-20`) |
| Tiếng Anh (conf ≥ 0.75) | Parakeet-CTC-110M (`csukuangfj/sherpa-onnx-nemo-parakeet_tdt_ctc_110m-en-36000`) |
| Tiếng Trung (conf ≥ 0.75) | SenseVoice (`iic/SenseVoiceSmall`) |
| Mixed / low-confidence | Whisper large-v3 (`Systran/faster-whisper-large-v3`, language=None) |

**Lưu ý trước khi chạy:**
- Bật **GPU** và **Internet** trong Kaggle
- Lưu `HF_TOKEN` và `OPENAI_API_KEY` (nếu cần tóm tắt) vào **Kaggle Secrets**
- Chạy Cell 1 (Install) → restart kernel → chạy từ Cell 2 trở đi


In [ ]:
import sys, subprocess, os

def _run(cmd, check=True):
    print(f"\n$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout or "") + (r.stderr or "")
    if out.strip():
        print(out[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"FAILED (exit {r.returncode}): {cmd}")

# ── Kaggle T4 x2 (Feb 2025+): Python 3.11, CUDA 12.4, torch 2.5.1+cu124 ──
# DO NOT reinstall torch — Kaggle wheels are cu124.
#
# [C1] DeepFilterNet==0.5.6 requires numpy<2.0.
#      Kaggle ships numpy 2.1.3, so pip will DOWNGRADE numpy to 1.26.4.
#      This is intentional and safe: torch 2.5.1 works with numpy 1.x.
#      pip 24.x rejects --no-deps when existing env violates constraints,
#      so we install normally and accept the numpy downgrade.
#
# [C2] funasr==1.3.7 has no pre-built cp311 wheel → pip builds from sdist
#      (~1–2 min). Pulls transformers==5.9.0 + tokenizers==0.22.2,
#      replacing Kaggle's pre-installed versions. Intentional.
# ─────────────────────────────────────────────────────────────────────────

# Step 1: DeepFilterNet — pip downgrades numpy 2.1.3 → 1.26.4 automatically
_run(f"{sys.executable} -m pip install -q DeepFilterNet==0.5.6 DeepFilterLib==0.5.6")

# Step 2: sherpa-onnx (Zipformer vi + Parakeet en)
_run(f"{sys.executable} -m pip install -q sherpa-onnx==1.13.2")

# Step 3: funasr (SenseVoice zh) — builds from sdist, takes ~1–2 min
_run(f"{sys.executable} -m pip install -q funasr==1.3.7")

# Step 4: faster-whisper (LID + Whisper fallback) + utilities
_run(f"{sys.executable} -m pip install -q faster-whisper==1.2.1 'soundfile>=0.12.1' 'openai>=1.0.0' 'huggingface_hub>=0.24.0' 'datasets>=2.0.0'")

# Verify imports
import importlib
print("\nVerifying imports:")
for pkg, label in [("df","DeepFilterNet"), ("sherpa_onnx","sherpa-onnx"),
                   ("funasr","funasr"), ("faster_whisper","faster-whisper"),
                   ("soundfile","soundfile"), ("openai","openai")]:
    try:
        importlib.import_module(pkg); print(f"  [OK] {label}")
    except ImportError as e:
        print(f"  [FAIL] {label}: {e}")

print("\n✓ All done. Restart kernel once, then run from Cell 2 onward.")

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    print("Batch mode — skipping restart.")
else:
    os.kill(os.getpid(), 9)


In [ ]:
from pathlib import Path
import os, sys, torch, time
from datetime import datetime

# ── Input / Output ────────────────────────────────────────────────────────
INPUT_MEDIA = "/kaggle/input/datasets/nguynthiknino/file-cuoc-hop-misa/cuoc_hop_1.m4a"

OUTPUT_DIR  = Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Secrets ───────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _sc = UserSecretsClient()
    HF_TOKEN       = _sc.get_secret("HF_TOKEN")
    OPENAI_API_KEY = _sc.get_secret("OPENAI_API_KEY")
except Exception:
    HF_TOKEN       = os.environ.get("HF_TOKEN", "")
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
if not HF_TOKEN:
    print("[WARN] HF_TOKEN empty — some model downloads may be rate-limited.")

# ── Pipeline parameters ───────────────────────────────────────────────────
DENOISE_CHUNK_SECONDS  = 30     # DeepFilterNet chunk size
MAX_CHUNK_DURATION     = 25.0   # max ASR chunk (seconds)
MAX_GAP                = 1.5    # max silence gap to merge VAD segments
MIN_CHUNK_DURATION     = 2.0    # drop chunks shorter than this
LID_CONFIDENCE         = 0.75   # below this → Whisper fallback with lang=None
SUMMARIZE              = bool(OPENAI_API_KEY)
SUMMARIZE_MODEL        = "gpt-4o-mini"

print("CUDA:", torch.cuda.is_available(),
      f"| GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "")
print("Input :", INPUT_MEDIA)
print("Output:", OUTPUT_DIR)
print("Summarize:", SUMMARIZE)

PIPELINE_T0 = time.perf_counter()
PIPELINE_STARTED_AT = datetime.now()
print("Started at:", PIPELINE_STARTED_AT.strftime("%Y-%m-%d %H:%M:%S"))


In [ ]:
# ── Preprocessing & Enhancement ──────────────────────────────────────────
import math, subprocess, shutil, tempfile
import torch, torchaudio
from df.enhance import enhance, init_df

def normalize_audio(input_path: str, output_wav: str):
    subprocess.run([
        "ffmpeg", "-y", "-i", input_path,
        "-ac", "1", "-ar", "48000",
        "-af", "loudnorm", "-c:a", "pcm_s16le", output_wav,
    ], check=True, capture_output=True)

class NoiseReducer:
    def __init__(self, chunk_seconds=30):
        print("Loading DeepFilterNet3...")
        self.model, self.df_state, _ = init_df()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.chunk_seconds = chunk_seconds
        print(f"DeepFilterNet ready on {self.device}")

    def process(self, input_wav: str, output_wav: str):
        audio, sr = torchaudio.load(input_wav)
        cs = sr * self.chunk_seconds
        total = audio.shape[1]
        chunks = []
        for i in range(math.ceil(total / cs)):
            s, e = i * cs, min((i + 1) * cs, total)
            chunk = audio[:, s:e].to(self.device)
            with torch.no_grad():
                chunks.append(enhance(self.model, self.df_state, chunk).cpu())
            del chunk
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        enhanced = torch.cat(chunks, dim=1)
        torchaudio.save(output_wav, enhanced, sr)

print("Preprocessing module ready.")


In [ ]:
# ── VAD Segmenter (Silero) ────────────────────────────────────────────────
from dataclasses import dataclass
import torch, torchaudio

@dataclass
class AudioChunk:
    start: float
    end: float

    @property
    def duration(self): return self.end - self.start

class VADSegmenter:
    TARGET_SR = 16_000

    def __init__(self, min_speech_ms=300, min_silence_ms=500,
                 max_chunk_duration=25.0, max_gap=1.5, min_chunk_duration=2.0):
        self.min_speech_ms      = min_speech_ms
        self.min_silence_ms     = min_silence_ms
        self.max_chunk_duration = max_chunk_duration
        self.max_gap            = max_gap
        self.min_chunk_duration = min_chunk_duration
        print("Loading Silero VAD...")
        self.model, utils = torch.hub.load(
            "snakers4/silero-vad", "silero_vad", trust_repo=True)
        self._get_ts = utils[0]
        print("Silero VAD ready.")

    def get_chunks(self, audio_path: str) -> list:
        wav, sr = torchaudio.load(audio_path)
        if sr != self.TARGET_SR:
            wav = torchaudio.functional.resample(wav, sr, self.TARGET_SR)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0)
        timestamps = self._get_ts(
            wav, self.model, sampling_rate=self.TARGET_SR,
            min_speech_duration_ms=self.min_speech_ms,
            min_silence_duration_ms=self.min_silence_ms,
            return_seconds=True,
        )
        segments = [AudioChunk(t["start"], t["end"]) for t in timestamps]
        chunks = self._merge(segments)
        print(f"VAD: {len(segments)} segment(s) → {len(chunks)} ASR chunk(s).")
        return chunks

    def _merge(self, segments):
        if not segments: return []
        chunks, c_start, c_end = [], segments[0].start, segments[0].end
        for seg in segments[1:]:
            if (seg.start - c_end) <= self.max_gap and (seg.end - c_start) <= self.max_chunk_duration:
                c_end = seg.end
            else:
                chunks.append(AudioChunk(c_start, c_end))
                c_start, c_end = seg.start, seg.end
        chunks.append(AudioChunk(c_start, c_end))

        merged, i = [], 0
        while i < len(chunks):
            ch = chunks[i]
            if ch.duration < self.min_chunk_duration:
                gap_prev = (ch.start - merged[-1].end) if merged else float("inf")
                gap_next = (chunks[i+1].start - ch.end) if i+1 < len(chunks) else float("inf")
                if gap_prev <= gap_next and merged:
                    merged[-1] = AudioChunk(merged[-1].start, ch.end)
                elif i+1 < len(chunks):
                    chunks[i+1] = AudioChunk(ch.start, chunks[i+1].end)
                else:
                    merged.append(ch)
            else:
                merged.append(ch)
            i += 1
        return merged

print("VAD module ready.")


In [ ]:
# ── LID + ASR Transcribers + Router ──────────────────────────────────────
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
import io, os, tempfile, numpy as np
import torch, torchaudio, soundfile as sf
from huggingface_hub import hf_hub_download

SUPPORTED_LANGS   = {"vi", "en", "zh"}
LID_THRESHOLD     = LID_CONFIDENCE   # from Config cell
LID_WINDOW_SEC    = 5.0
LID_SR            = 16_000

@dataclass
class TranscribedSegment:
    start: float; end: float; text: str
    language: str = ""; speaker: str = ""
    words: list = field(default_factory=list)

@dataclass
class LIDResult:
    language: str; confidence: float; is_mixed: bool

class BaseTranscriber(ABC):
    @abstractmethod
    def transcribe_chunk(self, audio_path, start, end, language=None): ...

# ── LID ──────────────────────────────────────────────────────────────────
class LangDetector:
    def __init__(self, whisper_model):
        self.model = whisper_model

    def detect(self, audio_path, start, end) -> LIDResult:
        wav, sr = torchaudio.load(audio_path)
        if sr != LID_SR:
            wav = torchaudio.functional.resample(wav, sr, LID_SR)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        s = int(start * LID_SR)
        e = int(min(start + LID_WINDOW_SEC, end) * LID_SR)
        audio = wav[0, s:e].numpy().astype(np.float32)
        lang, prob, _ = self.model.detect_language(audio)
        is_mixed = (prob < LID_THRESHOLD) or (lang not in SUPPORTED_LANGS)
        return LIDResult(lang, prob, is_mixed)

# ── Fallback: Whisper large-v3 ────────────────────────────────────────────
class FallbackTranscriber(BaseTranscriber):
    MODEL = "Systran/faster-whisper-large-v3"
    SR    = 16_000

    def __init__(self):
        from faster_whisper import WhisperModel
        device  = "cuda" if torch.cuda.is_available() else "cpu"
        compute = "float16" if device == "cuda" else "int8"
        print(f"Loading Whisper large-v3 on {device} ({compute})...")
        self.model = WhisperModel(self.MODEL, device=device, compute_type=compute)
        print("FallbackTranscriber ready.")

    def transcribe_chunk(self, audio_path, start, end, language=None):
        buf = self._buf(audio_path, start, end)
        if buf is None:
            return TranscribedSegment(start, end, "", language or "")
        segs, info = self.model.transcribe(buf, language=language, beam_size=5, vad_filter=False)
        text = " ".join(s.text.strip() for s in segs)
        return TranscribedSegment(start, end, text, info.language or (language or ""))

    def _buf(self, audio_path, start, end):
        wav, sr = torchaudio.load(audio_path)
        if sr != self.SR:
            wav = torchaudio.functional.resample(wav, sr, self.SR)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        chunk = wav[0, int(start*self.SR):int(end*self.SR)]
        if chunk.numel() == 0: return None
        buf = io.BytesIO()
        sf.write(buf, chunk.numpy().astype(np.float32), self.SR, format="WAV")
        buf.seek(0)
        return buf

# ── Vietnamese: Zipformer (sherpa-onnx) ───────────────────────────────────
class ViTranscriber(BaseTranscriber):
    HF_REPO = "csukuangfj/sherpa-onnx-zipformer-vi-int8-2025-04-20"
    FILES   = {"encoder": "encoder-epoch-12-avg-8.int8.onnx",
               "decoder": "decoder-epoch-12-avg-8.onnx",
               "joiner":  "joiner-epoch-12-avg-8.int8.onnx",
               "tokens":  "tokens.txt"}
    SR = 16_000

    def __init__(self):
        import sherpa_onnx
        print(f"Downloading Zipformer-vi from {self.HF_REPO}...")
        p = {k: hf_hub_download(self.HF_REPO, v) for k, v in self.FILES.items()}
        self.recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
            encoder=p["encoder"], decoder=p["decoder"], joiner=p["joiner"],
            tokens=p["tokens"], num_threads=4, provider="cpu")
        print("ViTranscriber ready.")

    def transcribe_chunk(self, audio_path, start, end, language="vi"):
        import sherpa_onnx
        samples = self._load(audio_path, start, end)
        if samples is None:
            return TranscribedSegment(start, end, "", "vi")
        stream = self.recognizer.create_stream()
        stream.accept_waveform(self.SR, samples)
        self.recognizer.decode_stream(stream)
        return TranscribedSegment(start, end, stream.result.text.strip(), "vi")

    def _load(self, audio_path, start, end):
        wav, sr = torchaudio.load(audio_path)
        if sr != self.SR:
            wav = torchaudio.functional.resample(wav, sr, self.SR)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        chunk = wav[0, int(start*self.SR):int(end*self.SR)]
        return chunk.numpy().astype(np.float32) if chunk.numel() else None

# ── English: Parakeet-CTC (sherpa-onnx) ──────────────────────────────────
class EnTranscriber(BaseTranscriber):
    HF_REPO = "csukuangfj/sherpa-onnx-nemo-parakeet_tdt_ctc_110m-en-36000"
    FILES   = {"model": "model.onnx", "tokens": "tokens.txt"}
    SR = 16_000

    def __init__(self):
        import sherpa_onnx
        print(f"Downloading Parakeet-CTC from {self.HF_REPO}...")
        p = {k: hf_hub_download(self.HF_REPO, v) for k, v in self.FILES.items()}
        self.recognizer = sherpa_onnx.OfflineRecognizer.from_nemo_ctc(
            model=p["model"], tokens=p["tokens"], num_threads=4, provider="cpu")
        print("EnTranscriber ready.")

    def transcribe_chunk(self, audio_path, start, end, language="en"):
        import sherpa_onnx
        samples = self._load(audio_path, start, end)
        if samples is None:
            return TranscribedSegment(start, end, "", "en")
        stream = self.recognizer.create_stream()
        stream.accept_waveform(self.SR, samples)
        self.recognizer.decode_stream(stream)
        return TranscribedSegment(start, end, stream.result.text.strip(), "en")

    def _load(self, audio_path, start, end):
        wav, sr = torchaudio.load(audio_path)
        if sr != self.SR:
            wav = torchaudio.functional.resample(wav, sr, self.SR)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        chunk = wav[0, int(start*self.SR):int(end*self.SR)]
        return chunk.numpy().astype(np.float32) if chunk.numel() else None

# ── Chinese: SenseVoice (FunASR) ─────────────────────────────────────────
class ZhTranscriber(BaseTranscriber):
    MODEL = "iic/SenseVoiceSmall"
    SR = 16_000

    def __init__(self):
        from funasr import AutoModel
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading SenseVoice on {device}...")
        self.model = AutoModel(model=self.MODEL, trust_remote_code=True, device=device)
        print("ZhTranscriber ready.")

    def transcribe_chunk(self, audio_path, start, end, language="zh"):
        tmp = self._extract(audio_path, start, end)
        try:
            res = self.model.generate(input=tmp, language=language or "auto", use_itn=True)
            text = res[0]["text"].strip() if res else ""
        finally:
            os.unlink(tmp)
        return TranscribedSegment(start, end, text, language or "zh")

    def _extract(self, audio_path, start, end):
        wav, sr = torchaudio.load(audio_path)
        if sr != self.SR:
            wav = torchaudio.functional.resample(wav, sr, self.SR)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        chunk = wav[:, int(start*self.SR):int(end*self.SR)]
        f = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        torchaudio.save(f.name, chunk, self.SR); f.close()
        return f.name

# ── ASR Router ────────────────────────────────────────────────────────────
class ASRRouter:
    def __init__(self):
        self._fallback = None
        self._lid      = None
        self._models   = {}

    def transcribe(self, audio_path, chunks):
        results = []
        for i, chunk in enumerate(chunks):
            lid = self._get_lid().detect(audio_path, chunk.start, chunk.end)
            tag = " [mixed→fallback]" if lid.is_mixed else ""
            print(f"  [{i+1}/{len(chunks)}] {chunk.start:.1f}s–{chunk.end:.1f}s | "
                  f"lang={lid.language} conf={lid.confidence:.2f}{tag}")
            seg = self._dispatch(audio_path, chunk, lid)
            if seg.text:
                results.append(seg)
        return sorted(results, key=lambda s: s.start)

    def _dispatch(self, audio_path, chunk, lid):
        if lid.is_mixed:
            return self._get_fallback().transcribe_chunk(audio_path, chunk.start, chunk.end, language=None)
        if lid.language in {"vi", "en", "zh"}:
            return self._get_specialized(lid.language).transcribe_chunk(
                audio_path, chunk.start, chunk.end, language=lid.language)
        return self._get_fallback().transcribe_chunk(audio_path, chunk.start, chunk.end, language=lid.language)

    def _get_fallback(self):
        if not self._fallback: self._fallback = FallbackTranscriber()
        return self._fallback

    def _get_lid(self):
        if not self._lid: self._lid = LangDetector(self._get_fallback().model)
        return self._lid

    def _get_specialized(self, lang):
        if lang not in self._models:
            self._models[lang] = {"vi": ViTranscriber, "en": EnTranscriber, "zh": ZhTranscriber}[lang]()
        return self._models[lang]

print("LID + ASR router modules ready.")


In [ ]:
# ── Formatter + Summarizer ───────────────────────────────────────────────
import json as _json, os as _os
from openai import OpenAI

def _srt_ts(sec):
    h = int(sec//3600); m = int((sec%3600)//60)
    s = int(sec%60);    ms = int((sec%1)*1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

class Formatter:
    def __init__(self, segments): self.segs = sorted(segments, key=lambda s: s.start)

    def to_text(self):
        return "\n".join(
            f"[{s.start:.2f}s–{s.end:.2f}s]{(' '+s.speaker+':') if s.speaker else ''} {s.text}"
            for s in self.segs)

    def to_plain_transcript(self):
        if not self.segs: return ""
        lines, cur_spk, cur_txt = [], self.segs[0].speaker, [self.segs[0].text]
        for s in self.segs[1:]:
            if s.speaker == cur_spk: cur_txt.append(s.text)
            else:
                lines.append(f"{cur_spk}: {' '.join(cur_txt)}" if cur_spk else ' '.join(cur_txt))
                cur_spk, cur_txt = s.speaker, [s.text]
        lines.append(f"{cur_spk}: {' '.join(cur_txt)}" if cur_spk else ' '.join(cur_txt))
        return "\n".join(lines)

    def to_srt(self):
        blocks = []
        for i, s in enumerate(self.segs, 1):
            blocks.append(f"{i}\n{_srt_ts(s.start)} --> {_srt_ts(s.end)}\n{(s.speaker+': ') if s.speaker else ''}{s.text}")
        return "\n\n".join(blocks)

    def to_json(self):
        return _json.dumps(
            [{"speaker": s.speaker, "start": s.start, "end": s.end,
              "text": s.text, "language": s.language} for s in self.segs],
            ensure_ascii=False, indent=2)

    def save(self, path, fmt="txt"):
        ext = _os.path.splitext(path)[1].lstrip(".")
        fmt = ext if ext in {"txt","srt","json"} else fmt
        content = {"txt": self.to_text, "plain": self.to_plain_transcript,
                   "srt": self.to_srt, "json": self.to_json}[fmt]()
        with open(path, "w", encoding="utf-8") as f: f.write(content)
        print(f"Saved → {path}")

_SUMMARY_PROMPT = """Bạn là trợ lý tóm tắt cuộc họp chuyên nghiệp.
Tóm tắt transcript sau bằng tiếng Việt với các phần:
1. **Tổng quan** (2–3 câu)
2. **Các điểm chính**
3. **Quyết định** (nếu có)
4. **Hành động tiếp theo** (nếu có)
Chỉ dùng nội dung trong transcript, không suy đoán."""

class Summarizer:
    def __init__(self, api_key=None, model="gpt-4o-mini"):
        self.model  = model
        self.client = OpenAI(api_key=api_key or _os.environ.get("OPENAI_API_KEY"))

    def summarize(self, transcript):
        if not transcript.strip(): raise ValueError("Empty transcript.")
        print(f"Summarizing with {self.model}...")
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "system", "content": _SUMMARY_PROMPT},
                      {"role": "user", "content": f"Transcript:\n\n{transcript}"}],
            temperature=0.3,
        )
        usage = resp.usage
        print(f"Tokens — prompt: {usage.prompt_tokens}, completion: {usage.completion_tokens}")
        return resp.choices[0].message.content.strip()

    def save(self, summary, path):
        with open(path, "w", encoding="utf-8") as f: f.write(summary)
        print(f"Summary saved → {path}")

print("Formatter + Summarizer ready.")


In [ ]:
# ── AudioPipeline ────────────────────────────────────────────────────────
import os, shutil, tempfile

class AudioPipeline:
    def __init__(self, chunk_seconds=30, max_chunk_duration=25.0, max_gap=1.5,
                 min_chunk_duration=2.0, openai_api_key=None, summarize_model="gpt-4o-mini"):
        self.noise_reducer = NoiseReducer(chunk_seconds=chunk_seconds)
        self.segmenter     = VADSegmenter(max_chunk_duration=max_chunk_duration,
                                          max_gap=max_gap,
                                          min_chunk_duration=min_chunk_duration)
        self.router        = ASRRouter()
        self.summarizer    = Summarizer(api_key=openai_api_key, model=summarize_model) if openai_api_key else None

    def run(self, input_file: str, output_dir: str) -> dict:
        os.makedirs(output_dir, exist_ok=True)
        base    = os.path.splitext(os.path.basename(input_file))[0]
        results = {}

        with tempfile.TemporaryDirectory() as tmp:
            norm_wav    = os.path.join(tmp, "normalized.wav")
            denoised_wav = os.path.join(tmp, "denoised.wav")

            print("[1] Normalizing...")
            normalize_audio(input_file, norm_wav)

            print("[2] Denoising...")
            self.noise_reducer.process(norm_wav, denoised_wav)

            audio_dir    = os.path.join(output_dir, "audio")
            os.makedirs(audio_dir, exist_ok=True)
            denoised_out = os.path.join(audio_dir, f"{base}_denoised.wav")
            shutil.copy(denoised_wav, denoised_out)
            results["denoised_wav"] = denoised_out

            print("[3] VAD segmentation...")
            chunks = self.segmenter.get_chunks(denoised_wav)
            if not chunks:
                print("No speech detected."); return results

            print(f"[4] Transcribing {len(chunks)} chunk(s) (LID + ASR routing)...")
            transcribed = self.router.transcribe(denoised_wav, chunks)

            fmt = Formatter(transcribed)
            for ext in ("txt", "json", "srt"):
                p = os.path.join(output_dir, f"{base}_transcript.{ext}")
                fmt.save(p, fmt=ext)
                results[f"transcript_{ext}"] = p

            if self.summarizer:
                print("[5] Summarizing...")
                summary_path = os.path.join(output_dir, f"{base}_summary.md")
                self.summarizer.save(self.summarizer.summarize(fmt.to_plain_transcript()), summary_path)
                results["summary_path"] = summary_path
            else:
                print("[5] Skipping summarization (no OPENAI_API_KEY).")

        print("\nDone. Outputs:")
        for k, v in results.items():
            print(f"  {k}: {v}")
        return results

print("AudioPipeline ready.")


In [ ]:
# ── Run Pipeline ─────────────────────────────────────────────────────────
pipeline = AudioPipeline(
    chunk_seconds      = DENOISE_CHUNK_SECONDS,
    max_chunk_duration = MAX_CHUNK_DURATION,
    max_gap            = MAX_GAP,
    min_chunk_duration = MIN_CHUNK_DURATION,
    openai_api_key     = OPENAI_API_KEY if SUMMARIZE else None,
    summarize_model    = SUMMARIZE_MODEL,
)

results = pipeline.run(INPUT_MEDIA, str(OUTPUT_DIR))


In [ ]:
# ── Results ──────────────────────────────────────────────────────────────
import time
from datetime import datetime

elapsed = time.perf_counter() - PIPELINE_T0
h, rem  = divmod(int(elapsed), 3600)
m, s    = divmod(rem, 60)

print(f"Total runtime: {h:02d}:{m:02d}:{s:02d}")
print(f"Started:  {PIPELINE_STARTED_AT.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

txt_path = results.get("transcript_txt")
if txt_path:
    print(f"\n{'─'*60}")
    print(open(txt_path, encoding="utf-8").read())
